# TAC-LAnoBERT v2 Training (2 Epochs)

**Purpose**: Train TAC-LAnoBERT v2 with all improvements (2 epochs for fair comparison)

**Steps**:
1. Setup environment
2. Split dataset
3. Preprocess & extract timestamps
4. Train tokenizer
5. Train TAC v2 model (2 epochs)
6. Evaluate results
7. Compare with baseline

**Config**: `configs/bgl_tac_v2_2epochs.yaml`

**v2 Improvements**:
- Early detection loss (penalty for late detection)
- Temporal features (7 types)
- Data augmentation (5 methods)
- Improved scoring (adaptive alpha, PCA)

**GPU Required**: T4 (~3-4 hours) or P100 (~2-3 hours)

**CPU**: Works but slow (~8-10 hours)

## 1. Setup Environment

In [ ]:
# Clone repository (if on Kaggle/Colab)
import os
if not os.path.exists('TAC-LAnoBERT-y'):
    !git clone https://github.com/rubyhcm/TAC-LAnoBERT-y.git
    %cd TAC-LAnoBERT-y
else:
    print("✅ Repository already exists")
    if not os.getcwd().endswith('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

In [ ]:
# Verify environment
import torch
import transformers
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  Running on CPU (will be slow)")
print("\n✅ Environment ready")

## 2. Check/Prepare BGL Data

In [ ]:
# Check if BGL.log exists, or link from Kaggle dataset
import os
import glob

bgl_log = "data/BGL/BGL.log"

if os.path.exists(bgl_log):
    size_mb = os.path.getsize(bgl_log) / (1024*1024)
    print(f"✅ BGL.log exists: {size_mb:.1f} MB")
else:
    print("BGL.log not found. Searching in Kaggle input...")
    os.makedirs("data/BGL", exist_ok=True)
    
    # Search for BGL.log in Kaggle input
    bgl_logs = glob.glob("/kaggle/input/**/BGL.log", recursive=True)
    if not bgl_logs:
        bgl_logs = glob.glob("/kaggle/input/**/*BGL*.log", recursive=True)
    
    if bgl_logs:
        target_log = bgl_logs[0]
        print(f"Found: {target_log}")
        os.system(f"ln -sf {target_log} {bgl_log}")
        print(f"✅ Linked to {bgl_log}")
    else:
        print("\n❌ BGL.log not found!")
        print("   On Kaggle: Click 'Add Data' → search 'BGL log'")
        print("   Local: Download BGL.log to data/BGL/")
        raise FileNotFoundError("BGL.log")

## 3. Split Dataset

Split BGL.log into train/test using TAC-LAnoBERT splitter

In [ ]:
# Split dataset
train_raw = "data/BGL/BGL_train_normal.raw"
test_raw = "data/BGL/BGL_test.raw"

if os.path.exists(train_raw) and os.path.exists(test_raw):
    print("✅ Dataset already split")
    print(f"   Train: {os.path.getsize(train_raw)/(1024*1024):.1f} MB")
    print(f"   Test:  {os.path.getsize(test_raw)/(1024*1024):.1f} MB")
else:
    print("Splitting dataset...")
    !python -m tac_lanobert.split_tac --config configs/bgl_tac_v2_2epochs.yaml
    
    if os.path.exists(train_raw) and os.path.exists(test_raw):
        print("\n✅ Dataset split successful")
    else:
        raise RuntimeError("Dataset split failed")

## 4. Preprocess & Extract Timestamps

In [ ]:
# Preprocess training set + extract timestamps
train_parsed = "data/BGL/BGL_train_normal_parsed.log"
train_ts = "data/BGL/BGL_train_normal_parsed.timestamps"

if os.path.exists(train_parsed) and os.path.exists(train_ts):
    print("✅ Training data already preprocessed")
else:
    print("Preprocessing training set...")
    !python -m tac_lanobert.preprocess_tac \
        --config configs/bgl_tac_v2_2epochs.yaml \
        --split train \
        --extract_timestamps
    
    if os.path.exists(train_ts):
        with open(train_ts) as f:
            count = sum(1 for _ in f)
        print(f"\n✅ Train timestamps: {count:,} lines")
    else:
        raise FileNotFoundError(train_ts)

In [ ]:
# Preprocess test set + extract timestamps
test_parsed = "data/BGL/BGL_test_parsed.log"
test_ts = "data/BGL/BGL_test_parsed.timestamps"

if os.path.exists(test_parsed) and os.path.exists(test_ts):
    print("✅ Test data already preprocessed")
else:
    print("Preprocessing test set...")
    !python -m tac_lanobert.preprocess_tac \
        --config configs/bgl_tac_v2_2epochs.yaml \
        --split test \
        --extract_timestamps
    
    if os.path.exists(test_ts):
        with open(test_ts) as f:
            count = sum(1 for _ in f)
        print(f"\n✅ Test timestamps: {count:,} lines")
    else:
        raise FileNotFoundError(test_ts)

## 5. Train Tokenizer

In [ ]:
# Train tokenizer
vocab_file = "outputs/BGL_tac_v2_2epochs/tokenizer/BGL_LogBERT-vocab.txt"

if os.path.exists(vocab_file):
    print("✅ Tokenizer already trained")
else:
    print("Training tokenizer...")
    !python -m tac_lanobert.tokenizer_tac --config configs/bgl_tac_v2_2epochs.yaml
    
    if os.path.exists(vocab_file):
        print(f"\n✅ Tokenizer saved to {vocab_file}")
    else:
        raise FileNotFoundError(vocab_file)

## 6. View Configuration

In [ ]:
# View config
import yaml

with open('configs/bgl_tac_v2_2epochs.yaml') as f:
    config = yaml.safe_load(f)

print("="*70)
print("TAC-LAnoBERT v2 CONFIGURATION")
print("="*70)

print(f"\nDataset: {config['dataset']}")
print(f"Run name: {config['run_name']}")

print(f"\n📝 Training:")
print(f"   Epochs: {config['train']['num_train_epochs']}")
print(f"   Batch size: {config['train']['per_device_train_batch_size']}")
print(f"   Learning rate: {config['train']['learning_rate']}")

print(f"\n⏰ TAC Features:")
tac = config.get('tac', {})
print(f"   Time2Vec: {tac.get('time2vec', {}).get('enabled', False)}")
print(f"   Memory Queue: {tac.get('memory', {}).get('enabled', False)}")

if 'tac_v2' in config:
    print(f"\n🚀 v2 Improvements:")
    v2 = config['tac_v2']
    print(f"   Early detection loss: {v2.get('early_detection_loss', {}).get('enabled', False)}")
    print(f"   Temporal features: {v2.get('temporal_features', {}).get('enabled', False)}")
    if v2.get('temporal_features', {}).get('enabled'):
        feats = v2['temporal_features'].get('features', [])
        print(f"      Features: {len(feats)}")
        for f in feats:
            print(f"        • {f}")
    print(f"   Data augmentation: {v2.get('data_augmentation', {}).get('enabled', False)}")

print("\n" + "="*70)

## 7. Train TAC-LAnoBERT v2 🚀

**Training**: 2 epochs (same as baseline for fair comparison)

**Time**: ~3-4 hours on GPU (T4), ~8-10 hours on CPU

**Why 2 epochs?**
- Same as original TAC-LAnoBERT baseline
- Fair comparison (same training budget)
- Verify improvements come from features, not more training

Make sure:
- ✅ GPU is enabled (Kaggle: Settings → Accelerator → GPU)
- ✅ Session won't timeout
- ✅ Internet connection stable

In [ ]:
import time
start_time = time.time()

print("="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("This will take ~3-4 hours on GPU...")
print("\n")

# Train TAC v2
!python -m tac_lanobert.train_tac --config configs/bgl_tac_v2_2epochs.yaml

end_time = time.time()
duration = end_time - start_time
hours = int(duration // 3600)
minutes = int((duration % 3600) // 60)

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"End time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {hours}h {minutes}m")
print("="*70)

## 8. Inference & Evaluation

In [ ]:
# Run inference
print("Running inference on test set...")
!python -m tac_lanobert.inference_tac --config configs/bgl_tac_v2_2epochs.yaml

print("\n✅ Inference complete")
print("   Check results in outputs/BGL_tac_v2_2epochs/results/")

## 9. View Results

In [ ]:
# Load and display results
import json
import numpy as np
from pathlib import Path

results_dir = Path('outputs/BGL_tac_v2_2epochs/results')

# Check for score files
score_files = list(results_dir.glob('scores_*.npy'))
if score_files:
    print("="*70)
    print("RESULTS")
    print("="*70)
    
    for score_file in sorted(score_files):
        scores = np.load(score_file)
        print(f"\n{score_file.name}:")
        print(f"  Shape: {scores.shape}")
        print(f"  Min: {scores.min():.4f}")
        print(f"  Max: {scores.max():.4f}")
        print(f"  Mean: {scores.mean():.4f}")
        print(f"  Std: {scores.std():.4f}")
    
    # Check for metrics file
    metrics_file = results_dir / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        
        print(f"\n📊 Metrics:")
        for key, value in metrics.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
    print("\n" + "="*70)
else:
    print("⚠️  No score files found")
    print(f"   Expected in: {results_dir}")

## 10. Summary

In [ ]:
from datetime import datetime

model_dir = Path('outputs/BGL_tac_v2_2epochs/model')
results_dir = Path('outputs/BGL_tac_v2_2epochs/results')

print("="*70)
print("TRAINING SUMMARY")
print("="*70)

print(f"\n📂 Output:")
if model_dir.exists():
    checkpoints = list(model_dir.glob('checkpoint-*'))
    print(f"   ✅ Model: {model_dir}")
    print(f"      Checkpoints: {len(checkpoints)}")
else:
    print(f"   ❌ Model not found")

if results_dir.exists():
    score_files = list(results_dir.glob('scores_*.npy'))
    print(f"   ✅ Results: {results_dir}")
    print(f"      Score files: {len(score_files)}")
else:
    print(f"   ❌ Results not found")

print(f"\n✅ Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n📋 NEXT STEPS:")
print("   1. Review results above")
print("   2. Compare with baseline")
print("   3. Download model (if on Kaggle)")
print("   4. Deploy if targets met")

print("\n" + "="*70)